# Quick Comparison with 01_Data_Exploration_And_Preprocessing

This notebook reruns the core preprocessing logic from Notebook 01 with pandas 3.x compatible code and concise checkpoints.

Current checkpoints after preprocessing:

- Transcript + profile + portfolio are merged into a single dataframe: `merged_df`.
- `offer_completed` label is created with the intended rule: `offer_viewed == 1` and `offer_completed_event == 1`.
- Missing values are handled:
  - `age`: replace `118` with missing, then fill with median
  - `income`: fill with median
  - `gender`: fill with `U`
  - `days_since_registration`: fill with median
- `channels` are one-hot encoded into columns such as `web`, `email`, `mobile`, `social`.

Reference checkpoints in this notebook:
- `merged_df` shape: `(50637, 18)`
- `offer_completed` rate: `0.483`
- null-containing columns after preprocessing: `0`

## 1) Merge transcript, portfolio, and profile

> Goal: create one person-offer level table that combines event behavior with customer and offer attributes.

This section reads raw files, builds offer event indicators (`received`, `viewed`, `completed_event`), merges all sources, and removes informational offers before preprocessing.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)

# Read in the json files
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'

portfolio = pd.read_json(DATA_DIR / 'portfolio.json', orient='records', lines=True)
profile = pd.read_json(DATA_DIR / 'profile.json', orient='records', lines=True)
transcript = pd.read_json(DATA_DIR / 'transcript.json', orient='records', lines=True)

# Build person-offer event indicators from transcript
offer_events = ['offer received', 'offer viewed', 'offer completed']
offers_df = transcript[transcript['event'].isin(offer_events)].copy()
offers_df['offer'] = offers_df['value'].map(
    lambda x: x.get('offer id') if x.get('offer id') is not None else x.get('offer_id')
)
offers_df = offers_df[['person', 'offer', 'event']]

offers_df = (
    pd.get_dummies(offers_df, columns=['event'], prefix='', prefix_sep='')
    .groupby(['person', 'offer'], as_index=False)
    .max()
    .rename(columns=lambda c: c.replace(' ', '_'))
)

for col in ['offer_received', 'offer_viewed', 'offer_completed']:
    if col not in offers_df.columns:
        offers_df[col] = 0

# Keep raw completion event for later labeling
offers_df = offers_df.rename(columns={'offer_completed': 'offer_completed_event'})

# Merge transcript-derived table with profile + portfolio
merged_df = offers_df.merge(profile, how='left', left_on='person', right_on='id').drop(columns=['id'])
merged_df = merged_df.merge(portfolio, how='left', left_on='offer', right_on='id').drop(columns=['id'])

# Remove informational offers (no completion objective)
merged_df = merged_df[merged_df['offer_type'] != 'informational'].copy()

print('Section 1 done - merged_df (before missing-value handling) shape:', merged_df.shape)
merged_df.head()

Section 1 done - merged_df (before missing-value handling) shape: (50637, 14)


,person,offer,offer_completed_event,offer_received,offer_viewed,gender,age,became_member_on,income,reward,channels,difficulty,duration,offer_type
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,True,True,False,M,33,20170421,72000.0,2,"[web, email, mobile]",10,7,discount
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,True,True,True,M,33,20170421,72000.0,5,"[web, email, mobile, social]",5,5,bogo
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,True,True,True,M,33,20170421,72000.0,2,"[web, email, mobile, social]",10,10,discount
5,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,False,True,True,None,118,20180425,NaN,5,"[web, email, mobile, social]",5,5,bogo
6,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,True,True,True,O,40,20180109,57000.0,5,"[web, email]",20,10,discount


## 2) Clean data and create target label

> Goal: produce a consistent modeling table with no critical missing values and a clear binary outcome.

This section creates `offer_completed` using viewed+completed logic, derives `days_since_registration`, expands channel indicators, and imputes key missing fields (`age`, `income`, `gender`).

In [2]:
# Create final success label and handle missing values (safe to re-run)
merged_df['offer_completed'] = (
    (merged_df['offer_viewed'] == 1) & (merged_df['offer_completed_event'] == 1)
).astype(int)

if 'became_member_on' in merged_df.columns:
    member_date = pd.to_datetime(merged_df['became_member_on'], format='%Y%m%d', errors='coerce')
    reference_date = member_date.max()
    merged_df['days_since_registration'] = (reference_date - member_date).dt.days
    merged_df = merged_df.drop(columns=['became_member_on'])

if 'channels' in merged_df.columns:
    merged_df['channels'] = merged_df['channels'].apply(lambda x: x if isinstance(x, list) else [])
    channels = merged_df['channels'].explode().dropna().unique()
    for ch in channels:
        merged_df[ch] = merged_df['channels'].apply(lambda x: int(ch in x))
    merged_df = merged_df.drop(columns=['channels'])

merged_df['age'] = merged_df['age'].replace(118, np.nan)
merged_df['age'] = merged_df['age'].fillna(merged_df['age'].median())
merged_df['income'] = merged_df['income'].fillna(merged_df['income'].median())
merged_df['gender'] = merged_df['gender'].fillna('U')

if 'days_since_registration' in merged_df.columns:
    merged_df['days_since_registration'] = merged_df['days_since_registration'].fillna(
        merged_df['days_since_registration'].median()
).astype(int)

print('Section 2 done - final merged_df shape:', merged_df.shape)
print('offer_completed in columns:', 'offer_completed' in merged_df.columns)
print('offer_completed rate:', round(merged_df['offer_completed'].mean(), 4))
print('Null columns count:', int((merged_df.isna().sum() > 0).sum()))

Section 2 done - final merged_df shape: (50637, 18)
offer_completed in columns: True
offer_completed rate: 0.483
Null columns count: 0


In [3]:
merged_df

,person,offer,offer_completed_event,offer_received,offer_viewed,gender,age,income,reward,difficulty,duration,offer_type,offer_completed,days_since_registration,web,email,mobile,social
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,True,True,False,M,33.0,72000.0,2,10,7,discount,0,461,1,1,1,0
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,True,True,True,M,33.0,72000.0,5,5,5,bogo,1,461,1,1,1,1
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,True,True,True,M,33.0,72000.0,2,10,10,discount,1,461,1,1,1,1
5,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,False,True,True,U,55.0,64000.0,5,5,5,bogo,0,92,1,1,1,1
6,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,True,True,True,O,40.0,57000.0,5,20,10,discount,1,198,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63283,fffad4f4828548d1b5583907f2e9906b,f19421c1d4aa40978ebb69ca19b0e20d,True,True,True,M,34.0,34000.0,5,5,5,bogo,1,549,1,1,1,1
63284,ffff82501cea40309d5fdd7edcca4a07,0b1e1539f2cc45b7b9fa7c272da2e1d7,True,True,True,F,45.0,62000.0,5,20,10,discount,1,608,1,1,0,0
63285,ffff82501cea40309d5fdd7edcca4a07,2906b810c7d4411798c6938adc9daaa5,True,True,True,F,45.0,62000.0,2,10,7,discount,1,608,1,1,1,0
63286,ffff82501cea40309d5fdd7edcca4a07,9b98b8c7a33c4b65b9aebfe6a799e6d9,True,True,True,F,45.0,62000.0,5,5,7,bogo,1,608,1,1,1,0


## 3) Build model-ready dataset and export

> Goal: generate the final preprocessed file used by downstream modeling notebooks.

This section keeps viewed offers, one-hot encodes selected categorical features, runs a duplicate check, and writes the final file to `data/processed/preprocessed_data.csv`.

In [4]:
# Keep only offers that were viewed, matching the training setup in Notebook 01
model_df = merged_df[merged_df['offer_viewed'] == 1].copy()

# Optional consistency check: no duplicate person-offer pairs after preprocessing
dup_count = int(model_df.duplicated(subset=['person', 'offer']).sum())

# Build model-ready features similar to Notebook 01 export step
encoded_df = pd.get_dummies(model_df, columns=['gender', 'offer_type'], drop_first=True)

output_dir = DATA_DIR / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'preprocessed_data.csv'
encoded_df.to_csv(output_path, index=False)

print('Section 3 done - model_df shape (viewed offers only):', model_df.shape)
print('Duplicate person-offer rows:', dup_count)
print('Encoded dataframe shape:', encoded_df.shape)
print('Saved:', output_path)

Section 3 done - model_df shape (viewed offers only): (39826, 18)
Duplicate person-offer rows: 0
Encoded dataframe shape: (39826, 20)
Saved: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\processed\preprocessed_data.csv


## Why This Version Is Stronger

- The workflow is split into clear sections, making it easier to validate and debug.
- Path handling is robust across notebook working directories by deriving `PROJECT_ROOT` and `DATA_DIR`.
- Event columns are validated before labeling to avoid pipeline breaks when categories are missing.
- The next section extends preprocessing with model-ready steps from Notebook 01 (view-filtering, encoding, and export).